# 🛡️ MPLADS AI Risk Intelligence Engine — Colab Notebook
### Smart India Hackathon Prototype — Data Engineering + ML Pipeline

This notebook is **Part 1 of 2** of the prototype:

| Part | Where it runs | What it does |
|---|---|---|
| **1. This Colab notebook** | Google Colab | Loads raw MPLADS export CSVs → cleans data → engineers risk features → trains an Isolation Forest anomaly model → combines it with a rule engine → exports risk-scored CSVs |
| **2. Streamlit app** (separate file, `streamlit_app/app.py`) | Local machine / Streamlit Cloud | Reads the exported CSVs from this notebook and renders the dark-themed interactive dashboard for officials |

**Run the cells in this notebook top to bottom.** At the end you will download 3 files
(`mp_risk_scores.csv`, `work_risk_scores.csv`, `vendor_features.csv`) — copy them into
`streamlit_app/data/` before running the Streamlit app.

> ⚠️ **Responsible-AI note:** This system flags **statistically anomalous patterns that
> deserve human verification**. It never labels a project, MP, or vendor as fraudulent
> or corrupt. All final judgement rests with officials who verify the flagged items.


## Step 1 — Install & import required libraries

In [3]:
# Step 1.1 — Install libraries not pre-installed on Colab (scikit-learn & pandas ARE pre-installed,
# this is just to guarantee matching versions across environments)
!pip install -q pandas==2.2.3 scikit-learn plotly joblib openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 79.1 MB/s eta 0:00:00


In [4]:
# Step 1.2 — Core imports used throughout this notebook
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("Libraries loaded successfully.")


Libraries loaded successfully.


## Step 2 — Upload the 4 official MPLADS export CSVs

Download these directly from the official MPLADS/eSAKSHI dashboard
(https://mplads.mospi.gov.in/digigov/dashboard.html) and upload them here:

1. **Completed works** export
2. **Expenditures** export
3. **MP summary** export
4. **Recommended works** export


In [6]:
# Step 2.1 — File upload widget (Colab-only). If you are running this notebook
# outside Colab, comment this cell out and set the 4 paths manually in Step 2.2 instead.
try:
    from google.colab import files
    print("Please select and upload the 4 MPLADS CSV files (multi-select is allowed):")
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    print("Uploaded:", uploaded_names)
except ImportError:
    uploaded_names = []
    print("Not running inside Google Colab — skip to Step 2.2 and set file paths manually.")


Please select and upload the 4 MPLADS CSV files (multi-select is allowed):


Saving mplads_mp_summary_2026-09-14.csv to mplads_mp_summary_2026-09-14.csv
Saving mplads_expenditures_2026-09-14.csv to mplads_expenditures_2026-09-14.csv
Saving mplads_completed_works_2026-09-14.csv to mplads_completed_works_2026-09-14.csv
Saving mplads_recommended_works_2026-09-14.csv to mplads_recommended_works_2026-09-14.csv
Uploaded: ['mplads_mp_summary_2026-09-14.csv', 'mplads_expenditures_2026-09-14.csv', 'mplads_completed_works_2026-09-14.csv', 'mplads_recommended_works_2026-09-14.csv']


In [7]:
# Step 2.2 — Map uploaded filenames to their role.
# If your filenames differ from the defaults below, edit this dictionary.
import re

def find_file(keyword, names):
    for n in names:
        if keyword in n.lower():
            return n
    return None

COMPLETED_PATH   = find_file("completed", uploaded_names)   or "mplads_completed_works_2026-09-13.csv"
EXPENDITURE_PATH = find_file("expenditure", uploaded_names) or "mplads_expenditures_2026-09-10.csv"
MP_SUMMARY_PATH  = find_file("summary", uploaded_names)     or "mplads_mp_summary_2026-09-10.csv"
RECOMMENDED_PATH = find_file("recommended", uploaded_names) or "mplads_recommended_works_2026-09-10.csv"

print("Completed works file :", COMPLETED_PATH)
print("Expenditure file     :", EXPENDITURE_PATH)
print("MP summary file      :", MP_SUMMARY_PATH)
print("Recommended works    :", RECOMMENDED_PATH)


Completed works file : mplads_completed_works_2026-09-14.csv
Expenditure file     : mplads_expenditures_2026-09-14.csv
MP summary file      : mplads_mp_summary_2026-09-14.csv
Recommended works    : mplads_recommended_works_2026-09-14.csv


## Step 3 — Load and inspect the raw data (sanity check)

In [8]:
# Step 3.1 — Load each CSV as-is and look at shape + a few rows.
# This is a pure sanity check before any cleaning happens.
raw_completed   = pd.read_csv(COMPLETED_PATH)
raw_expenditure = pd.read_csv(EXPENDITURE_PATH)
raw_mp_summary  = pd.read_csv(MP_SUMMARY_PATH)
raw_recommended = pd.read_csv(RECOMMENDED_PATH)

print("Completed works   :", raw_completed.shape)
print("Expenditures      :", raw_expenditure.shape)
print("MP summary        :", raw_mp_summary.shape)
print("Recommended works :", raw_recommended.shape)
raw_completed.head(3)


Completed works   : (44028, 12)
Expenditures      : (108695, 10)
MP summary        : (774, 16)
Recommended works : (87272, 11)


,Work ID,Work Description,Category,MP Name,Constituency,State,House,Final Amount (₹),Completed Date,Has Images,Average Rating,IDA
0,134703,Upgradation of Road from Madhavaram Village to...,Normal/Others,DAGGUMALLA PRASADA RAO,CHITTOOR,Andhra Pradesh,Lok Sabha,499993.0,2025-01-31T00:00:00.000Z,True,NaN,CHITTOOR(DISTRICT COLLECTOR CHITTOOR_IDA)
1,135593,Construction of CC Road from Amudala Village t...,Normal/Others,DAGGUMALLA PRASADA RAO,CHITTOOR,Andhra Pradesh,Lok Sabha,448722.0,2024-12-05T00:00:00.000Z,True,NaN,CHITTOOR(DISTRICT COLLECTOR CHITTOOR_IDA)
2,135595,Construction of CC road from Amudala Village t...,Normal/Others,DAGGUMALLA PRASADA RAO,CHITTOOR,Andhra Pradesh,Lok Sabha,448970.0,2024-12-05T00:00:00.000Z,True,NaN,CHITTOOR(DISTRICT COLLECTOR CHITTOOR_IDA)


In [9]:
# Step 3.2 — Quick null-value / dtype check (helps catch encoding issues early)
for name, df in [("completed", raw_completed), ("expenditure", raw_expenditure),
                  ("mp_summary", raw_mp_summary), ("recommended", raw_recommended)]:
    print(f"--- {name} ---")
    print(df.isna().sum()[df.isna().sum() > 0])
    print()


--- completed ---
Work Description       85
Category                5
Average Rating      44024
dtype: int64

--- expenditure ---
Series([], dtype: int64)

--- mp_summary ---
Average Rating    770
dtype: int64

--- recommended ---
Work Description    52
Category             5
dtype: int64



## Step 4 — Write the shared Risk Engine module

This is the **same module** used by the Streamlit app (`risk_engine.py`), written to disk
here with `%%writefile` so every line is visible and auditable. It contains:

- `load_raw_tables()` — cleaning & type parsing
- `build_vendor_features()` — vendor concentration signals from the expenditure ledger
- `build_completed_features()` — transparency & cost-outlier signals from completed works
- `score_mp_level()` — the hybrid rule + Isolation Forest risk model for MPs
- `score_work_level()` — the risk model for individual completed works
- `run_full_pipeline()` — orchestrates everything end-to-end


In [10]:
%%writefile risk_engine.py
"""
=====================================================================
 MPLADS AI Risk Intelligence Engine
 ---------------------------------------------------------------
 Shared core logic used by BOTH:
   1) The Google Colab notebook (training / batch scoring pipeline)
   2) The Streamlit dashboard (serving / display layer)

 IMPORTANT DESIGN PRINCIPLE
 ---------------------------------------------------------------
 This engine NEVER labels a project "fraudulent" or "corrupt".
 It only computes an explainable RISK / ANOMALY score that tells
 officials WHERE to look first and WHY, based on statistical
 deviation from normal MPLADS patterns. All outputs are meant to
 trigger human verification, not automated judgement.
=====================================================================
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

RUPEE = "₹"

# ---------------------------------------------------------------
# 0. LOADING & CLEANING
# ---------------------------------------------------------------

def load_raw_tables(completed_path, expenditure_path, mp_summary_path, recommended_path):
    """Load the four official MPLADS export CSVs and do light type cleaning."""
    completed = pd.read_csv(completed_path)
    expenditure = pd.read_csv(expenditure_path)
    mp_summary = pd.read_csv(mp_summary_path)
    recommended = pd.read_csv(recommended_path)

    # Strip whitespace from column names (defensive - some exports have trailing spaces)
    for df in (completed, expenditure, mp_summary, recommended):
        df.columns = [c.strip() for c in df.columns]

    # Parse dates
    completed["Completed Date"] = pd.to_datetime(completed["Completed Date"], errors="coerce")
    expenditure["Expenditure Date"] = pd.to_datetime(expenditure["Expenditure Date"], errors="coerce")
    recommended["Recommendation Date"] = pd.to_datetime(recommended["Recommendation Date"], errors="coerce")

    # Normalise MP name / constituency casing & whitespace to make joins reliable
    for df in (completed, expenditure, mp_summary, recommended):
        for col in ("MP Name", "Constituency", "State", "IDA"):
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()

    return completed, expenditure, mp_summary, recommended


# ---------------------------------------------------------------
# 1. VENDOR-LEVEL FEATURES  (built from the expenditure ledger)
# ---------------------------------------------------------------

def build_vendor_features(expenditure: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate the expenditure ledger to one row per MP, capturing vendor
    concentration and payment-pattern signals. Expenditure rows do not carry
    a Work ID (the 'Work Description' field there is a scheme *category*,
    not a specific project) so vendor analysis is done at MP level.
    """
    ex = expenditure.copy()
    ex["is_round_amount"] = (ex["Expenditure Amount (₹)"] % 10000 == 0)
    ex["is_success"] = ex["Payment Status"].eq("Payment Success")

    rows = []
    for mp, g in ex.groupby(["MP Name", "Constituency", "State", "House"]):
        total_amt = g["Expenditure Amount (₹)"].sum()
        vendor_amt = g.groupby("Vendor")["Expenditure Amount (₹)"].sum().sort_values(ascending=False)
        top_vendor = vendor_amt.index[0] if len(vendor_amt) else None
        top_vendor_share = (vendor_amt.iloc[0] / total_amt * 100) if total_amt > 0 and len(vendor_amt) else 0
        # Herfindahl-Hirschman style concentration index (0-100 scale, 100 = single vendor monopoly)
        shares = (vendor_amt / total_amt) if total_amt > 0 else vendor_amt * 0
        hhi = float((shares ** 2).sum() * 100)

        # same vendor, same day, multiple payments -> possible invoice splitting
        same_day_multi = (
            g.groupby(["Vendor", "Expenditure Date"]).size().gt(1).sum()
        )

        rows.append({
            "MP Name": mp[0], "Constituency": mp[1], "State": mp[2], "House": mp[3],
            "num_vendors": g["Vendor"].nunique(),
            "num_transactions": len(g),
            "total_expenditure_ledger": total_amt,
            "top_vendor": top_vendor,
            "top_vendor_share_pct": round(top_vendor_share, 2),
            "vendor_hhi": round(hhi, 2),
            "round_amount_txn_pct": round(ex_round := (g["is_round_amount"].mean() * 100), 2),
            "payment_inprogress_pct": round((1 - g["is_success"].mean()) * 100, 2),
            "same_day_same_vendor_clusters": int(same_day_multi),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# 2. COMPLETED-WORKS TRANSPARENCY FEATURES (aggregated to MP level)
# ---------------------------------------------------------------

def build_completed_features(completed: pd.DataFrame) -> pd.DataFrame:
    cw = completed.copy()
    cw["Category"] = cw["Category"].fillna("Unknown")

    # Cost outlier z-score within each work Category (peer-group comparison)
    cat_stats = cw.groupby("Category")["Final Amount (₹)"].agg(["mean", "std"]).rename(
        columns={"mean": "cat_mean", "std": "cat_std"})
    cw = cw.merge(cat_stats, on="Category", how="left")
    cw["cat_std"] = cw["cat_std"].replace(0, np.nan)
    cw["cost_zscore"] = (cw["Final Amount (₹)"] - cw["cat_mean"]) / cw["cat_std"]
    cw["cost_zscore"] = cw["cost_zscore"].fillna(0)

    # Duplicate / near-duplicate description reused by the same MP
    dup_counts = cw.groupby(["MP Name", "Work Description"]).size().rename("desc_repeat_count")
    cw = cw.merge(dup_counts, on=["MP Name", "Work Description"], how="left")

    # Exact same rupee amount repeated many times by the same MP (cookie-cutter billing)
    amt_counts = cw.groupby(["MP Name", "Final Amount (₹)"]).size().rename("amount_repeat_count")
    cw = cw.merge(amt_counts, on=["MP Name", "Final Amount (₹)"], how="left")

    cw["is_round_amount"] = (cw["Final Amount (₹)"] % 50000 == 0)
    cw["no_images"] = ~cw["Has Images"].astype(bool)
    cw["no_rating"] = cw["Average Rating"].isna()

    agg = cw.groupby(["MP Name", "Constituency", "State", "House"]).agg(
        completed_count=("Work ID", "count"),
        avg_cost_zscore=("cost_zscore", "mean"),
        high_cost_outlier_count=("cost_zscore", lambda s: int((s > 2).sum())),
        no_image_pct=("no_images", lambda s: round(s.mean() * 100, 2)),
        no_rating_pct=("no_rating", lambda s: round(s.mean() * 100, 2)),
        round_amount_pct=("is_round_amount", lambda s: round(s.mean() * 100, 2)),
        max_desc_repeat=("desc_repeat_count", "max"),
        max_amount_repeat=("amount_repeat_count", "max"),
    ).reset_index()

    return agg, cw  # return both MP-level rollup and enriched per-work table


# ---------------------------------------------------------------
# 3. MP-LEVEL RISK MODEL  (rule engine + Isolation Forest anomaly score)
# ---------------------------------------------------------------

RULE_WEIGHTS = {
    "utilization_completion_gap": 20,
    "vendor_monopoly": 18,
    "high_pending_payments": 12,
    "low_transparency_images": 10,
    "low_transparency_rating": 8,
    "high_round_amount": 8,
    "cost_outliers": 12,
    "duplicate_descriptions": 8,
    "repeated_exact_amounts": 8,
    "large_unpaid_balance": 10,
}


def score_mp_level(mp_summary, vendor_feats, completed_rollup):
    df = mp_summary.merge(vendor_feats, on=["MP Name", "Constituency", "State", "House"], how="left")
    df = df.merge(completed_rollup, on=["MP Name", "Constituency", "State", "House"], how="left")

    # Fill NA for MPs with no expenditure / no completed works yet
    fill_cols = ["num_vendors", "num_transactions", "top_vendor_share_pct", "vendor_hhi",
                 "round_amount_txn_pct", "payment_inprogress_pct", "same_day_same_vendor_clusters",
                 "completed_count", "avg_cost_zscore", "high_cost_outlier_count", "no_image_pct",
                 "no_rating_pct", "round_amount_pct", "max_desc_repeat", "max_amount_repeat"]
    for c in fill_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    df["pending_payment_ratio"] = np.where(
        df["Transaction Count"] > 0, df["Pending Payments"] / df["Transaction Count"] * 100, 0)

    df["util_completion_gap"] = (df["Utilization %"] - df["Completion Rate %"]).clip(lower=0)

    df["unpaid_balance_pct"] = np.where(
        df["Total Expenditure (₹)"] > 0,
        df["Balance Not Yet Paid to Vendors (₹)"] / df["Total Expenditure (₹)"] * 100, 0)

    # ---------------- Rule triggers (boolean) + reasons text -----------------
    reasons_col, checklist_col, score_col = [], [], []

    for _, r in df.iterrows():
        pts = 0
        reasons = []
        checklist = []

        if r["util_completion_gap"] >= 40:
            pts += RULE_WEIGHTS["utilization_completion_gap"]
            reasons.append(
                f"Funds utilization ({r['Utilization %']:.0f}%) is far ahead of physical completion "
                f"({r['Completion Rate %']:.0f}%) — a gap of {r['util_completion_gap']:.0f} points.")
            checklist.append("Verify that advance/interim payments correspond to actual physical progress on site.")

        if r["vendor_hhi"] >= 40 and r["num_transactions"] >= 5:
            pts += RULE_WEIGHTS["vendor_monopoly"]
            reasons.append(
                f"Vendor concentration is high (HHI={r['vendor_hhi']:.0f}); "
                f"top vendor '{r.get('top_vendor','-')}' handles {r['top_vendor_share_pct']:.0f}% of spend.")
            checklist.append("Check whether the dominant vendor was selected through fair/competitive process across multiple works.")

        if r["pending_payment_ratio"] >= 25 and r["Transaction Count"] >= 5:
            pts += RULE_WEIGHTS["high_pending_payments"]
            reasons.append(f"{r['pending_payment_ratio']:.0f}% of payment transactions are still pending.")
            checklist.append("Review pending-payment vendor invoices for delay reasons and documentation completeness.")

        if r["no_image_pct"] >= 60 and r["completed_count"] >= 3:
            pts += RULE_WEIGHTS["low_transparency_images"]
            reasons.append(f"{r['no_image_pct']:.0f}% of completed works have no geo/photo evidence uploaded.")
            checklist.append("Request site photographs / geo-tagged images for works missing visual evidence.")

        if r["no_rating_pct"] >= 90 and r["completed_count"] >= 3:
            pts += RULE_WEIGHTS["low_transparency_rating"]
            reasons.append("Almost none of the completed works have received citizen/beneficiary feedback ratings.")
            checklist.append("Cross-check beneficiary feedback mechanism / conduct spot citizen verification.")

        if r["round_amount_pct"] >= 50 and r["completed_count"] >= 3:
            pts += RULE_WEIGHTS["high_round_amount"]
            reasons.append(f"{r['round_amount_pct']:.0f}% of completed works are billed in suspiciously round figures.")
            checklist.append("Ask for itemised cost estimates/bills rather than lump-sum round amounts.")

        if r["high_cost_outlier_count"] >= 2:
            pts += RULE_WEIGHTS["cost_outliers"]
            reasons.append(
                f"{int(r['high_cost_outlier_count'])} completed works cost far above the average for their "
                f"work category (statistical outliers).")
            checklist.append("Compare quoted cost of outlier works with category benchmark / schedule of rates (SOR).")

        if r["max_desc_repeat"] >= 4:
            pts += RULE_WEIGHTS["duplicate_descriptions"]
            reasons.append(f"The same work description text is reused {int(r['max_desc_repeat'])} times.")
            checklist.append("Verify these are genuinely distinct works and not duplicate/split entries.")

        if r["max_amount_repeat"] >= 4:
            pts += RULE_WEIGHTS["repeated_exact_amounts"]
            reasons.append(f"The exact same rupee amount appears on {int(r['max_amount_repeat'])} separate completed works.")
            checklist.append("Check for templated/cookie-cutter billing rather than work-specific costing.")

        if r["unpaid_balance_pct"] >= 40 and r["Total Expenditure (₹)"] > 0:
            pts += RULE_WEIGHTS["large_unpaid_balance"]
            reasons.append(f"{r['unpaid_balance_pct']:.0f}% of expenditure is still an unpaid balance to vendors.")
            checklist.append("Confirm vendor payment schedule and reasons for outstanding dues.")

        score_col.append(pts)
        reasons_col.append(reasons)
        checklist_col.append(checklist)

    df["rule_score_raw"] = score_col
    df["reasons"] = reasons_col
    df["checklist"] = checklist_col

    # ---------------- Isolation Forest anomaly score (unsupervised) -----------------
    feature_cols = [
        "Utilization %", "Completion Rate %", "util_completion_gap", "vendor_hhi",
        "top_vendor_share_pct", "pending_payment_ratio", "no_image_pct", "no_rating_pct",
        "round_amount_pct", "avg_cost_zscore", "unpaid_balance_pct", "same_day_same_vendor_clusters",
    ]
    X = df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
    scaler = MinMaxScaler()
    Xs = scaler.fit_transform(X)

    iso = IsolationForest(n_estimators=300, contamination=0.12, random_state=42)
    iso.fit(Xs)
    # decision_function: higher = more normal. Flip & scale to 0-100 (higher = more anomalous)
    raw_anomaly = -iso.decision_function(Xs)
    anomaly_scaled = MinMaxScaler((0, 100)).fit_transform(raw_anomaly.reshape(-1, 1)).ravel()
    df["ml_anomaly_score"] = anomaly_scaled.round(1)

    # ---------------- Hybrid final score -----------------
    rule_scaled = MinMaxScaler((0, 100)).fit_transform(df[["rule_score_raw"]]).ravel()
    df["rule_score_scaled"] = rule_scaled.round(1)
    df["risk_score"] = (0.6 * df["rule_score_scaled"] + 0.4 * df["ml_anomaly_score"]).round(1)

    def band(s):
        if s >= 70: return "High"
        if s >= 40: return "Medium"
        return "Low"
    df["risk_level"] = df["risk_score"].apply(band)

    df["reasons_text"] = df["reasons"].apply(lambda L: " | ".join(L) if L else "No significant rule triggered — flagged mainly by statistical pattern deviation." )
    df["checklist_text"] = df["checklist"].apply(lambda L: " | ".join(sorted(set(L))) if L else "General random-sample verification recommended.")

    return df.sort_values("risk_score", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------
# 4. WORK-LEVEL RISK MODEL (drill-down on individual completed works)
# ---------------------------------------------------------------

def score_work_level(cw_enriched: pd.DataFrame) -> pd.DataFrame:
    df = cw_enriched.copy()
    pts = np.zeros(len(df))
    reasons_list = [[] for _ in range(len(df))]

    cond = df["cost_zscore"] > 2
    pts += cond * 30
    for i in np.where(cond)[0]:
        reasons_list[i].append(
            f"Cost is a statistical outlier for its category (z-score={df['cost_zscore'].iloc[i]:.1f}).")

    cond = df["no_images"]
    pts += cond * 15
    for i in np.where(cond)[0]:
        reasons_list[i].append("No photographic/geo-tagged evidence uploaded for this work.")

    cond = df["no_rating"]
    pts += cond * 10
    for i in np.where(cond)[0]:
        reasons_list[i].append("No citizen/beneficiary rating recorded.")

    cond = df["desc_repeat_count"] >= 4
    pts += cond * 20
    for i in np.where(cond)[0]:
        reasons_list[i].append(
            f"Identical description reused {int(df['desc_repeat_count'].iloc[i])} times by this MP.")

    cond = df["amount_repeat_count"] >= 4
    pts += cond * 15
    for i in np.where(cond)[0]:
        reasons_list[i].append(
            f"Same exact amount seen on {int(df['amount_repeat_count'].iloc[i])} other works by this MP.")

    cond = df["is_round_amount"]
    pts += cond * 10
    for i in np.where(cond)[0]:
        reasons_list[i].append("Billed amount is a suspiciously round figure.")

    df["risk_points"] = pts
    df["risk_score"] = MinMaxScaler((0, 100)).fit_transform(df[["risk_points"]]).round(1)
    df["reasons_text"] = [" | ".join(r) if r else "No red flags — routine work." for r in reasons_list]

    def band(s):
        if s >= 70: return "High"
        if s >= 35: return "Medium"
        return "Low"
    df["risk_level"] = df["risk_score"].apply(band)
    return df.sort_values("risk_score", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------
# 5. END-TO-END PIPELINE
# ---------------------------------------------------------------

def run_full_pipeline(completed_path, expenditure_path, mp_summary_path, recommended_path):
    completed, expenditure, mp_summary, recommended = load_raw_tables(
        completed_path, expenditure_path, mp_summary_path, recommended_path)

    vendor_feats = build_vendor_features(expenditure)
    completed_rollup, cw_enriched = build_completed_features(completed)

    mp_risk = score_mp_level(mp_summary, vendor_feats, completed_rollup)
    work_risk = score_work_level(cw_enriched)

    return {
        "mp_risk": mp_risk,
        "work_risk": work_risk,
        "vendor_feats": vendor_feats,
        "raw": {
            "completed": completed, "expenditure": expenditure,
            "mp_summary": mp_summary, "recommended": recommended,
        }
    }


Writing risk_engine.py


## Step 5 — Run the pipeline step by step (with explanations)

Instead of calling `run_full_pipeline()` as one black box, we call each stage separately
so you can inspect intermediate outputs — exactly what a judge / evaluator will want to see.


In [11]:
# Step 5.1 — Reload the module fresh (in case you re-ran Step 4 after editing it)
import importlib
import risk_engine
importlib.reload(risk_engine)
from risk_engine import (
    load_raw_tables, build_vendor_features, build_completed_features,
    score_mp_level, score_work_level,
)


In [12]:
# Step 5.2 — Load & clean the 4 tables (dates parsed, whitespace stripped)
completed, expenditure, mp_summary, recommended = load_raw_tables(
    COMPLETED_PATH, EXPENDITURE_PATH, MP_SUMMARY_PATH, RECOMMENDED_PATH
)
print("Cleaned shapes:", completed.shape, expenditure.shape, mp_summary.shape, recommended.shape)


Cleaned shapes: (44028, 12) (108695, 10) (774, 16) (87272, 11)


In [13]:
# Step 5.3 — Build vendor-concentration features from the expenditure ledger
# (Herfindahl-Hirschman Index, top-vendor share, same-day-same-vendor payment clusters, etc.)
vendor_feats = build_vendor_features(expenditure)
vendor_feats.sort_values("vendor_hhi", ascending=False).head(10)


,MP Name,Constituency,State,House,num_vendors,num_transactions,total_expenditure_ledger,top_vendor,top_vendor_share_pct,vendor_hhi,round_amount_txn_pct,payment_inprogress_pct,same_day_same_vendor_clusters
332,Nishikant Dubey,GODDA,Jharkhand,Lok Sabha,1,1,1785510.0,PRAKHAND SHIKSHA PRASAR PADADHIKARI,100.00,100.00,0.00,0.0,0
205,Gajendra Singh Shekhawat,JODHPUR,Rajasthan,Lok Sabha,1,1,491827.0,Maa Bhadariya Rai Enterprises,100.00,100.00,0.00,0.0,0
564,Shri Paka Venkata Satyanarayana (2025-28),Sitting Rajya Sabha,Andhra Pradesh,Rajya Sabha,1,1,1599035.0,EXECUTIVE ENGINEER APCPDCL CHIRALA,100.00,100.00,0.00,0.0,0
532,Shri Kartikeya Sharma (2022-28),Sitting Rajya Sabha,Haryana,Rajya Sabha,1,2,1161226.0,KMT Construction Company,100.00,100.00,0.00,0.0,0
523,Shri Jogen Mohan (2026-32),Sitting Rajya Sabha,Assam,Rajya Sabha,1,1,1500000.0,CONST OF ROAD FROM B B LINKED ROAD TO UPEN DAS...,100.00,100.00,100.00,100.0,0
527,Shri Jyotirmay Singh Mahato,PURULIA,West Bengal,Lok Sabha,1,31,11711831.0,ATANU PATRA,100.00,100.00,0.00,0.0,3
612,Shrikant Eknath Shinde,KALYAN,Maharashtra,Lok Sabha,1,1,325000.0,Pro Ortho Perfect India Pvt Ltd,100.00,100.00,0.00,0.0,0
632,Smt. Adhikarimayum Sharda Devi (2026-32),Sitting Rajya Sabha,Manipur,Rajya Sabha,1,1,1800000.0,Akin construction and supply,100.00,100.00,100.00,0.0,0
360,Prataprao Jadhav,BULDHANA,Maharashtra,Lok Sabha,1,2,1208028.0,SHARMA CONSTRUCTION,100.00,100.00,0.00,0.0,0
138,DR. PRABHA MALLIKARJUN,DAVANAGERE,Karnataka,Lok Sabha,3,90,57444696.0,NIRMITHI KENDRA DAVANGERE,97.83,95.74,31.11,0.0,19


In [14]:
# Step 5.4 — Build completed-works transparency & cost-outlier features
# (category peer-group cost z-score, duplicate description detection, repeated-amount detection)
completed_rollup, cw_enriched = build_completed_features(completed)
completed_rollup.sort_values("high_cost_outlier_count", ascending=False).head(10)


,MP Name,Constituency,State,House,completed_count,avg_cost_zscore,high_cost_outlier_count,no_image_pct,no_rating_pct,round_amount_pct,max_desc_repeat,max_amount_repeat
555,Shri S. Selvaganabathy (2021-27),Sitting Rajya Sabha,Puducherry,Rajya Sabha,31,1.788300,16,19.35,100.0,3.23,1.0,1
59,Asit Kumar Mal,BOLPUR,West Bengal,Lok Sabha,134,0.567404,16,11.19,100.0,5.22,2.0,12
215,Indra Hang Subba,SIKKIM,Sikkim,Lok Sabha,38,1.455715,15,65.79,100.0,21.05,2.0,4
352,Prof. Ram Gopal Yadav (2020-26),Sitting Rajya Sabha,Uttar Pradesh,Rajya Sabha,25,2.550122,14,0.00,100.0,0.00,1.0,1
493,Shri Jaggesh (2022-28),Sitting Rajya Sabha,Karnataka,Rajya Sabha,126,0.377949,14,63.49,100.0,13.49,9.0,7
133,DR. LATA WANKHEDE,SAGAR,Madhya Pradesh,Lok Sabha,103,0.214611,13,22.33,100.0,38.83,5.0,31
12,AKSHAYA YADAV,FIROZABAD,Uttar Pradesh,Lok Sabha,37,1.776949,11,48.65,100.0,0.00,1.0,1
455,Shri Arun Singh (2020-26),Sitting Rajya Sabha,Uttar Pradesh,Rajya Sabha,46,0.544074,11,0.00,100.0,0.00,1.0,22
612,Smt. S. Phangnon Konyak (2022-28),Sitting Rajya Sabha,Nagaland,Rajya Sabha,58,1.921993,11,15.52,100.0,94.83,2.0,21
600,Smt. Darshana Singh (2022-28),Sitting Rajya Sabha,Uttar Pradesh,Rajya Sabha,31,3.882613,10,3.23,100.0,0.00,3.0,2


In [15]:
# Step 5.5 — Score every MP (rule engine + Isolation Forest anomaly model, blended)
mp_risk = score_mp_level(mp_summary, vendor_feats, completed_rollup)
mp_risk[["MP Name", "State", "risk_score", "risk_level"]].head(15)


,MP Name,State,risk_score,risk_level
0,VISHWESHWAR HEGDE KAGERI,Karnataka,87.5,High
1,TANUJ PUNIA,Uttar Pradesh,78.8,High
2,DULU MAHATO,Jharkhand,78.7,High
3,Shri Jairam Ramesh (2022-28),Karnataka,76.1,High
4,Ram Shiromani,Uttar Pradesh,76.1,High
5,H.D. KUMARASWAMY,Karnataka,75.0,High
6,DR. RAJ KUMAR CHABBEWAL,Punjab,73.9,High
7,RAJESH RANJAN ALIAS PAPPU YADAV,Bihar,73.4,High
8,ADHIKARI SOUMENDU,West Bengal,72.7,High
9,Annpurna Devi,Jharkhand,72.6,High


In [16]:
# Step 5.6 — Score every individual completed work
work_risk = score_work_level(cw_enriched)
work_risk[["Work ID", "MP Name", "Final Amount (₹)", "risk_score", "risk_level", "reasons_text"]].head(10)


,Work ID,MP Name,Final Amount (₹),risk_score,risk_level,reasons_text
0,151023,Indra Hang Subba,3000000.0,100.0,High,Cost is a statistical outlier for its category...
1,151022,Indra Hang Subba,3000000.0,100.0,High,Cost is a statistical outlier for its category...
2,151021,Indra Hang Subba,3000000.0,100.0,High,Cost is a statistical outlier for its category...
3,151024,Indra Hang Subba,3000000.0,100.0,High,Cost is a statistical outlier for its category...
4,203467,DR. LATA WANKHEDE,2599420.0,93.8,High,Cost is a statistical outlier for its category...
5,201271,Shri Narayanasa K. Bhandage (2024-30),4995360.0,93.8,High,Cost is a statistical outlier for its category...
6,201272,Shri Narayanasa K. Bhandage (2024-30),4995360.0,93.8,High,Cost is a statistical outlier for its category...
7,205872,DR. LATA WANKHEDE,2599420.0,93.8,High,Cost is a statistical outlier for its category...
8,204268,Shri Narayanasa K. Bhandage (2024-30),4995360.0,93.8,High,Cost is a statistical outlier for its category...
9,203466,DR. LATA WANKHEDE,2599420.0,93.8,High,Cost is a statistical outlier for its category...


## Step 6 — Explore & validate the model output

In [17]:
# Step 6.1 — Overall risk-level distribution across MPs
dist = mp_risk["risk_level"].value_counts()
fig = px.pie(values=dist.values, names=dist.index, hole=0.5,
             color=dist.index,
             color_discrete_map={"High": "#f87171", "Medium": "#fbbf24", "Low": "#34d399"},
             title="MP risk-level distribution")
fig.show()


<!doctype html>

In [18]:
# Step 6.2 — Do the rule score and the ML anomaly score agree with each other?
# (If they diverge a lot, it usually means the ML model is catching something
#  the hand-written rules missed — worth inspecting those rows.)
fig = px.scatter(mp_risk, x="rule_score_scaled", y="ml_anomaly_score", color="risk_level",
                  color_discrete_map={"High": "#f87171", "Medium": "#fbbf24", "Low": "#34d399"},
                  hover_data=["MP Name", "State"],
                  title="Rule-based score vs. ML anomaly score per MP")
fig.show()


<!doctype html>

In [19]:
# Step 6.3 — Top 15 highest-risk MPs with their reasons (for a quick manual sanity read)
for _, r in mp_risk.head(15).iterrows():
    print(f"[{r['risk_level']:>6} | {r['risk_score']:>5.1f}] {r['MP Name']} ({r['State']})")
    print("   Why      :", r["reasons_text"])
    print("   Verify   :", r["checklist_text"])
    print()


[  High |  87.5] VISHWESHWAR HEGDE KAGERI (Karnataka)
   Why      : Funds utilization (66%) is far ahead of physical completion (21%) — a gap of 45 points. | Vendor concentration is high (HHI=89); top vendor 'ZILLA NIRMITi KENDRA' handles 94% of spend. | 83% of completed works have no geo/photo evidence uploaded. | Almost none of the completed works have received citizen/beneficiary feedback ratings. | 93% of completed works are billed in suspiciously round figures. | The exact same rupee amount appears on 15 separate completed works. | 177% of expenditure is still an unpaid balance to vendors.
   Verify   : Ask for itemised cost estimates/bills rather than lump-sum round amounts. | Check for templated/cookie-cutter billing rather than work-specific costing. | Check whether the dominant vendor was selected through fair/competitive process across multiple works. | Confirm vendor payment schedule and reasons for outstanding dues. | Cross-check beneficiary feedback mechanism / conduct spo

In [20]:
# Step 6.4 — State-wise average risk score (helps spot state-level patterns)
state_avg = mp_risk.groupby("State")["risk_score"].mean().sort_values(ascending=False)
fig = px.bar(state_avg, orientation="h", title="Average MP risk score by state")
fig.update_layout(yaxis=dict(autorange="reversed"), showlegend=False)
fig.show()


<!doctype html>

## Step 7 — Export the scored datasets for the Streamlit dashboard

We export 3 CSVs. Copy them into `streamlit_app/data/` (replacing the bundled sample files)
so the dashboard reflects your freshly-scored run.


In [21]:
# Step 7.1 — Trim work_risk to the columns the dashboard actually needs (keeps file small)
work_risk_export = work_risk[[
    "Work ID", "Work Description", "Category", "MP Name", "Constituency", "State", "House",
    "Final Amount (₹)", "Completed Date", "Has Images", "Average Rating", "IDA",
    "cost_zscore", "desc_repeat_count", "amount_repeat_count", "is_round_amount",
    "no_images", "no_rating", "risk_score", "risk_level", "reasons_text",
]]

# Step 7.2 — Drop the raw list-typed columns from mp_risk before export (CSV can't store lists cleanly)
mp_risk_export = mp_risk.drop(columns=["reasons", "checklist"])

# Step 7.3 — Write everything to disk
mp_risk_export.to_csv("mp_risk_scores.csv", index=False)
work_risk_export.to_csv("work_risk_scores.csv", index=False)
vendor_feats.to_csv("vendor_features.csv", index=False)

import json
meta = {
    "n_mps": len(mp_risk_export),
    "n_works": len(work_risk_export),
    "high_risk_mps": int((mp_risk_export.risk_level == "High").sum()),
    "high_risk_works": int((work_risk_export.risk_level == "High").sum()),
}
json.dump(meta, open("meta.json", "w"), indent=2)
print("Exported files:", "mp_risk_scores.csv, work_risk_scores.csv, vendor_features.csv, meta.json")
print(meta)


Exported files: mp_risk_scores.csv, work_risk_scores.csv, vendor_features.csv, meta.json
{'n_mps': 774, 'n_works': 44028, 'high_risk_mps': 13, 'high_risk_works': 1339}


In [22]:
# Step 7.4 — Download the exported files to your computer (Colab-only)
try:
    from google.colab import files
    for fname in ["mp_risk_scores.csv", "work_risk_scores.csv", "vendor_features.csv", "meta.json"]:
        files.download(fname)
except ImportError:
    print("Not running inside Colab — the files are already saved in the current working directory.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 8 — Next step

Copy the 4 downloaded files into `streamlit_app/data/` (overwrite the bundled sample files),
then run the dashboard:

```bash
cd streamlit_app
pip install -r requirements.txt
streamlit run app.py
```

See `README.md` in the project root for full local-run and deployment instructions
(Streamlit Community Cloud, Hugging Face Spaces, Render, etc.).
